# Рубежный контроль №1
## Вариант 8 (ИБМ3-64Б)

**Выполнил:** Зазовский В.Д.  
**Группа:** ИБМ3-64Б  
**Дата:** 2026

### Задание
**Задача №1:** Для заданного набора данных проведите корреляционный анализ. В случае наличия пропусков в данных удалите строки или колонки, содержащие пропуски. Сделайте выводы о возможности построения моделей машинного обучения и о возможном вкладе признаков в модель.

**Доп. требование:** для пары произвольных колонок данных построить график "Диаграмма рассеяния".

**Набор данных №8:** Boston Housing Dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
import warnings
warnings.filterwarnings('ignore')

# Загрузка датасета
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print("=== BOSTON HOUSING DATASET (California Housing) ===")
print(f"Размер: {df.shape}")
print(f"\nПризнаки: {list(df.columns)}")
print("\nПервые 10 строк:")
print(df.head(10))


In [ ]:
# Описание признаков
print("=== ОПИСАНИЕ ПРИЗНАКОВ ===")
print("MedInc: медианный доход в районе")
print("HouseAge: медианный возраст домов")
print("AveRooms: среднее количество комнат")
print("AveBedrms: среднее количество спален")
print("Population: население района")
print("AveOccup: среднее число жильцов")
print("Latitude: широта")
print("Longitude: долгота")
print("MedHouseVal: медианная стоимость дома (целевая переменная)")
print("\n=== СТАТИСТИЧЕСКОЕ ОПИСАНИЕ ===")
print(df.describe().round(3))


In [ ]:
# Проверка пропусков
print("=== ПРОПУСКИ В ДАННЫХ ===")
missing = df.isnull().sum()
print(missing)
print(f"\nВсего пропусков: {missing.sum()}")
if missing.sum() > 0:
    print("Пропуски найдены, удаляем строки с пропусками...")
    df_clean = df.dropna()
    print(f"Размер после удаления: {df_clean.shape}")
else:
    print("Пропусков нет!")
    df_clean = df.copy()


In [ ]:
# === КОРРЕЛЯЦИОННЫЙ АНАЛИЗ ===
plt.figure(figsize=(12, 10))
corr_matrix = df_clean.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Корреляционная матрица признаков', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/rk1/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== КОРРЕЛЯЦИЯ С ЦЕЛЕВОЙ ПЕРЕМЕННОЙ ===")
target_corr = corr_matrix['MedHouseVal'].drop('MedHouseVal').sort_values(key=abs, ascending=False)
print(target_corr.round(3))


In [ ]:
# === ДИАГРАММА РАССЕЯНИЯ ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(df_clean['MedInc'], df_clean['MedHouseVal'], alpha=0.3, s=10, c='steelblue')
axes[0].set_xlabel('MedInc (медианный доход)')
axes[0].set_ylabel('MedHouseVal (стоимость дома)')
axes[0].set_title('Диаграмма рассеяния: MedInc vs MedHouseVal', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

scatter = axes[1].scatter(df_clean['Longitude'], df_clean['Latitude'], 
                         c=df_clean['MedHouseVal'], cmap='viridis', alpha=0.5, s=10)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('Географическое распределение стоимости', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=axes[1], label='MedHouseVal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/rk1/scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

pair_corr = df_clean['MedInc'].corr(df_clean['MedHouseVal'])
print(f"\nКорреляция MedInc vs MedHouseVal: {pair_corr:.3f}")


In [ ]:
# === РАСПРЕДЕЛЕНИЯ ===
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.hist(df_clean['MedHouseVal'], bins=50, color='steelblue', edgecolor='black')
plt.xlabel('MedHouseVal'); plt.ylabel('Частота')
plt.title('Распределение целевой переменной', fontsize=11, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.hist(df_clean['MedInc'], bins=50, color='coral', edgecolor='black')
plt.xlabel('MedInc'); plt.ylabel('Частота')
plt.title('Распределение дохода', fontsize=11, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
top_corr_features = target_corr.head(5)
plt.barh(top_corr_features.index, top_corr_features.values, color='forestgreen')
plt.xlabel('Корреляция с MedHouseVal')
plt.title('Топ-5 признаков по корреляции', fontsize=11, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/rk1/distributions.png', dpi=150, bbox_inches='tight')
plt.show()


### Выводы

1. **Пропуски**: В наборе данных пропусков не обнаружено, все 20640 записей полные.

2. **Корреляция с целевой переменной**:
   - **MedInc (0.688)** — сильная положительная корреляция: доход района является ключевым фактором стоимости жилья.
   - **AveRooms (0.151)** — слабая положительная корреляция.
   - **HouseAge (0.105)** — очень слабая корреляция.
   - **Latitude (-0.144)** и **Longitude (-0.045)** — география влияет на стоимость.
   - **AveBedrms (-0.046)**, **Population (-0.025)**, **AveOccup (-0.021)** — практически не коррелируют.

3. **Мультиколлинеарность**: Сильная корреляция между AveRooms и AveBedrms (0.85).

4. **Возможность построения моделей**: Набор данных хорошо подходит для регрессии. Наличие сильного предиктора (MedInc) обеспечивает базовое качество модели.

5. **Рекомендации**: Использовать MedInc как основной признак, учесть мультиколлинеарность AveRooms/AveBedrms, применить масштабирование.
